# 04 · Alpaca ETF costs confirmed

One-shot audit of the locked `B_ETF_ALPACA` table for core universe B (`ETF_MULTI`). Writes
`04_backtest/s3_trend/artifacts/alpaca_etf_costs_confirmed.json`.


## 0. Imports & Config


In [ ]:
import json
import os
import sys
import warnings
from datetime import date

import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from data.ingestion.equity_fetcher import fetch_ohlcv
from data.processing.s3_fx_price_panel import ETF_MULTI_TICKERS
from execution.brokers.alpaca_broker import (
    S1_ALPACA_API_KEY,
    S1_ALPACA_SECRET_KEY,
    S2_ALPACA_API_KEY,
    S2_ALPACA_SECRET_KEY,
    AlpacaBroker,
)
from strategies.s3_trend.costs import (
    COSTS,
    ETF_COST_PROFILE,
    daily_borrow_return,
    leg_cost_bps,
)

PROFILE = ETF_COST_PROFILE
cfg = COSTS[PROFILE]
ARTIFACT = os.path.join(
    ROOT, "04_backtest", "s3_trend", "artifacts", "alpaca_etf_costs_confirmed.json"
)
print("profile", PROFILE)
print("leg_cost_bps", leg_cost_bps("SPY", 100.0, profile=PROFILE))


## 1. Locked table vs current mids


In [ ]:
rows = []
for ticker in ETF_MULTI_TICKERS:
    mid = float("nan")
    try:
        px = fetch_ohlcv(ticker, "2024-01-01", None, interval="1d")
        if px is not None and not px.empty:
            mid = float(px["close"].iloc[-1])
    except Exception as exc:
        warnings.warn(f"{ticker} mid fetch failed: {exc!r}")
    bps = (
        leg_cost_bps(ticker, mid, profile=PROFILE) if mid == mid else float("nan")
    )
    rows.append(
        {
            "ticker": ticker,
            "mid": mid,
            "commission_bps": cfg["commission_bps"],
            "third_party_fee_bps": cfg["third_party_fee_bps"],
            "slippage_bps_per_leg": cfg["slippage_bps_per_leg"],
            "leg_cost_bps": bps,
        }
    )
tbl = pd.DataFrame(rows)
display(tbl)
assert set(ETF_MULTI_TICKERS) == set(tbl["ticker"])
print("flat leg cost", cfg["commission_bps"] + cfg["third_party_fee_bps"] + cfg["slippage_bps_per_leg"])


## 2. Alpaca tradable / shortable / ETB flags


In [ ]:
short_rows = []
short_error = ""
try:
    try:
        broker = AlpacaBroker(
            paper=True,
            api_key_name=S1_ALPACA_API_KEY,
            secret_key_name=S1_ALPACA_SECRET_KEY,
        )
    except Exception:
        broker = AlpacaBroker(
            paper=True,
            api_key_name=S2_ALPACA_API_KEY,
            secret_key_name=S2_ALPACA_SECRET_KEY,
        )
    short_rows = broker.get_asset_shortability(list(ETF_MULTI_TICKERS))
except Exception as exc:
    short_error = str(exc)
    warnings.warn(f"Alpaca shortability unavailable ({exc!r}); continuing without live flags")
    short_rows = [
        {
            "ticker": t,
            "alpaca_symbol": "",
            "tradable": None,
            "shortable": None,
            "easy_to_borrow": None,
            "found": False,
            "error": short_error,
        }
        for t in ETF_MULTI_TICKERS
    ]
short_tbl = pd.DataFrame(short_rows)
display(short_tbl)
htb = short_tbl.loc[short_tbl["easy_to_borrow"] == False]
not_short = short_tbl.loc[short_tbl["shortable"] == False]
print("not ETB", list(htb["ticker"]) if not htb.empty else [])
print("not shortable", list(not_short["ticker"]) if not not_short.empty else [])


## 3. Example: $100k long SPY, 30 days


In [ ]:
notional = 100_000.0
weight = 1.0
spy_row = tbl.loc[tbl["ticker"] == "SPY"].iloc[0]
entry_exit_bps = 2.0 * float(spy_row["leg_cost_bps"])
trade_cost = notional * entry_exit_bps / 10_000.0
borrow_accrual = 0.0
asof = pd.Timestamp("2024-06-03")
for i in range(30):
    borrow_accrual += daily_borrow_return(PROFILE, weight=weight)
print("SPY mid", spy_row["mid"])
print("round-trip bps", entry_exit_bps)
print("round-trip $ on $100k", trade_cost)
print("30d borrow return (long)", borrow_accrual)
print("30d borrow $ on $100k", borrow_accrual * notional)


## 4. Write confirmation JSON


In [ ]:
payload = {
    "profile": PROFILE,
    "confirmed": True,
    "doc_date": str(date.today()),
    "commission_bps": cfg["commission_bps"],
    "third_party_fee_bps": cfg["third_party_fee_bps"],
    "slippage_bps_per_leg": cfg["slippage_bps_per_leg"],
    "borrow_bps_annual": cfg["borrow_bps_annual"],
    "trading_days_per_year": cfg["trading_days_per_year"],
    "leg_cost_bps": cfg["commission_bps"] + cfg["third_party_fee_bps"] + cfg["slippage_bps_per_leg"],
    "tickers": list(ETF_MULTI_TICKERS),
    "example_spy_rt_cost_usd_100k": trade_cost,
    "example_spy_30d_borrow_return_long": borrow_accrual,
    "shortability": short_rows,
    "shortability_error": short_error,
    "note": (
        "research lock — Alpaca retail API commission-free US ETFs; "
        "0.1 bps SEC/TAF/CAT RT-averaged; 3.2 bps open-fill slippage; "
        "ETB borrow 0; revisit if Alpaca schedule or HTB names diverge"
    ),
}
os.makedirs(os.path.dirname(ARTIFACT), exist_ok=True)
with open(ARTIFACT, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, default=str)
    f.write("\n")
print("wrote", ARTIFACT)
